_Células em Markdown foram traduzidas, cometários originais feitos nas células de código._

### KNN (K-Nearest-Neighbors)

KNN é um conceito simples: defina uma medida de distância entre itens do conjunto de dados, e encontre o _K_ próximos itens. Você pode utilizá-los para predizer algumas propriedades de um item de teste, através de um tipo de "votação" nele.

Por exemplo, vamos olhar um conjunto de filmes. Nós iremos tentar adivinhar a nota de um filme olhando os 10 (_K_) filmes mais próximos dele em termos de gênero e popularidade. 

Para começar, vamos carregar cada valiação do conjunto de dados em um `DataFrame` Pandas.

In [1]:
import pandas as pd

r_cols = ['user_id', 'movie_id', 'rating']
ratings = pd.read_csv('arquivos_anexo/u.data', sep='\t', names=r_cols, usecols=range(3))
ratings.head()

,user_id,movie_id,rating
0,0,50,5
1,0,172,5
2,0,133,1
3,196,242,3
4,186,302,3


Agora, vamos agrupar tudo pelo ID do filme, é calcular o total de avaliações (a popularidade de cada filme) e a
média de avaliação para cada filme:

In [2]:
import numpy as np

movieProperties = ratings.groupby('movie_id').agg({'rating': [np.size, np.mean]}) # Agrupa os filmes pelo ID do filme 
                                                                                    # ('movie_id') Vamos mostrar de 
                                                                                    # forma agregada a quantidade e a 
                                                                                    # média das avaliações.
movieProperties.head()

rating          
           size      mean
movie_id                 
1           452  3.878319
2           131  3.206107
3            90  3.033333
4           209  3.550239
5            86  3.302326

A quantidade de avaliações (`size`) não é muito útil para calcular a distância, então iremos criar um novo
`DataFrame` que contém as avaliações normalizadas. Assim o valor de 0 significa que ninguém avaliou e 1 indica
o filme mais popular.

In [3]:
movieNumRatings = pd.DataFrame(movieProperties['rating']['size']) # Acessa a coluna 'size' do agregado em 
                                                                    # `rating`
movieNormalizedNumRatings = movieNumRatings.apply(lambda x: (x - np.min(x)) / (np.max(x) - np.min(x)))
                                                    # (x - np.min(x)) -> significa a distância de x pro valor mínimo 
                                                    # do intervalo.
                                                    # (np.max(x) - np.min(x)) -> Define o tamanho do intervalo.
                                                    # Ao dividir um pelo outro medimos a diância de X entre o mais 
                                                    # popular  ou menos popular indicado por um valor entre 0 e 1 
                                                    # normalizando assim o dado.
                                                    # Lembrando que X é uma `Serie` e podemos realizar operações 
                                                    # escalares nas `Serie`s do Pandas.
movieNormalizedNumRatings.head()

,size
movie_id,
1,0.773585
2,0.222985
3,0.152659
4,0.356775
5,0.145798


Agora, vamos busar a informações sobre os genêros do arquivo `u.item`. A forma que isto funciona, há 19 campos, 
cada um corresponde a um gênero específico - o valor `0` significa que não é aquele gênero, e `1` significa que é 
determinado gênero. Um filme pode ter mais de um gênerio associado a ele.

Enquanto percorremos a informação do filme, vamos colocar tudo em um dicionário Python chamado `movieDict`. Cada valor irá conter 
o nome do filme, a lista de gêneros, a nota de popularidade normalizada, a média de cada avaliação para cada filme:

In [4]:
# Por conta do Erro que acontece nessa célula tive que mudar o código da aula conforme a célula abaixo
# comentado para não dar erro
'''
movieDict = {}
with open(r'arquivos_anexo/u.item') as f:
    temp = ''
    for line in f: # Percorre lendo cada linha do arquivo.
        fields = line.rstrip('\n').split('|')
        movieID = int(fields[0])
        name = fields[1]
        genres = fields[5:25]
        genres = map(int, genres)
        movieDict[movieID] = (name, genres, movieNormalizedNumRatings.loc[movieID].get('size'), movieProperties.loc[movieID].rating.loc['mean'])
'''

None

In [5]:
movieDict = {}
with open('arquivos_anexo/u.item', 'rb') as f: # Abre o arquivo para leitura
                                                # Tive que mudar o modo de leitura par `rb`, leitura em bytes [1].
    temp = ''
    for line in f: # Percorre lendo cada linha do arquivo.
        fields = line.decode('latin-1').rstrip('\n').split('|') # Transformei o arquivo em bytes para string utilizando
                                                                # `decode` [2]. Utilizei latin-1 por conta do erro do 
                                                                # utf-8 [3] ao iterar em determinada linha do arquivo.
        movieID = int(fields[0]) # Fields acaba se tornando uma lista e os elementos da lista representam os valores
                                    # dos dados
        name = fields[1]
        genres = fields[5:25]
        genres = list(map(int, genres)) # Transforma em uma lista de inteiros utiliando o `map` em seguida foi 
                                        # convertido o objeto para uma lista.
        movieDict[movieID] = (name, genres, movieNormalizedNumRatings.loc[movieID].get('size'), movieProperties.loc[movieID].rating.loc['mean'])

# Referências (acesso em 14/08/2026): 
# [1] https://stackoverflow.com/questions/33054527/typeerror-a-bytes-like-object-is-required-not-str-when-handling-file-conte
# [2] https://stackoverflow.com/questions/606191/convert-bytes-to-a-string-in-python-3
# [3] https://stackoverflow.com/questions/5552555/unicodedecodeerror-invalid-continuation-byte 

Por exemplo, aqui está o registro do filme com ID 1, "Toy Story":

In [6]:
movieDict[1]

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 np.float64(3.8783185840707963))

Agora vamos definir uma função que calcula a "distância" entre dois filmes baseado em quão semelhante são os 
seus gêneros e sua popularidade. Para ter certeza que funciona vamos calcular a distância entre os filmes com IDs 2 e 4:

In [7]:
from scipy import spatial

def ComputeDistance(a, b):
    genresA = a[1]
    genresB = b[1]
    genreDistance = spatial.distance.cosine(genresA, genresB) # Calcula a distância pelo cosseno dado os vetores dos 
                                                                # gêneros.
    popularityA = a[2]
    popularityB = b[2]
    popularityDistance = abs(popularityA - popularityB) # Para popularidade utiliza a diferença como disância.
    return genreDistance + popularityDistance # Soma as duas distâncias para ter a distância total. Popularidade é um
                                                # eixo e genêro é outro.

ComputeDistance(movieDict[2], movieDict[4]) # Quanto mais próximo a 0 o resultado mais parecidos são os filmes.

np.float64(0.8004574042309892)

Lembre-se quanto maior a distância menos parecidos são os filmes. Vamos conferir quais são os filmes 2 e 4 e 
confirmar que eles não são parecidos:

In [8]:
print(movieDict[2])
print(movieDict[4])

('GoldenEye (1995)', [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], np.float64(0.22298456260720412), np.float64(3.2061068702290076))
('Get Shorty (1995)', [0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], np.float64(0.3567753001715266), np.float64(3.550239234449761))


Agora estamos a poucos trechos de código para calcular a disância entre um filme utilizado para teste e todos
os filmes no conjunto de dados. Vamos ordenar por distância e mostrar os _KNNs_.

In [9]:
import operator

def getNeighbors(movieID, K):
    distances = []
    for movie in movieDict:
        if (movie != movieID): # Utilizado para não considerar o próprio filme no teste.
            dist = ComputeDistance(movieDict[movieID], movieDict[movie]) # Calcula a disância
            distances.append((movie, dist)) # Cria uma lista de tupla representando o filme e a sua distância

    distances.sort(key=operator.itemgetter(1)) # Ordena por distância (elemento 1 da tupla).
    neighbors = []
    for x in range(K): # Percorre os valores ordenados dado valor K (quantidade de vizinhos).
        neighbors.append(distances[x][0]) # Adiciona o filme na lista de vizinhos.
    return neighbors

K = 10
avgRating = 0
neighbors = getNeighbors(1,K)
for neighbor in neighbors:
    avgRating += movieDict[neighbor][3] # O elemento 3 do `dataset` de filmes representa a nota do filme.
    print(movieDict[neighbor][0] + " " + str(movieDict[neighbor][3]))

avgRating /= float(K) # Calcula a média do filme por K

Liar Liar (1997) 3.156701030927835
Aladdin (1992) 3.8127853881278537
Willy Wonka and the Chocolate Factory (1971) 3.6319018404907975
Monty Python and the Holy Grail (1974) 4.0664556962025316
Full Monty, The (1997) 3.926984126984127
George of the Jungle (1997) 2.685185185185185
Beavis and Butt-head Do America (1996) 2.7884615384615383
Birdcage, The (1996) 3.4436860068259385
Home Alone (1990) 3.0875912408759123
Aladdin and the King of Thieves (1996) 2.8461538461538463


Agora vamos visualizar qual foi a média calculada dos 10 KNNs de Toy Story:

In [10]:
avgRating

np.float64(3.3445905900235564)

Como isso se compara com a nota atual de Toy Story?

In [11]:
movieDict[1] # Uma diferença pequena considerando o calculo algoritmo feito, mas não é o melhor.

('Toy Story (1995)',
 [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 np.float64(0.7735849056603774),
 np.float64(3.8783185840707963))

### Atividade

A escolha de 10 K foi arbitrária - quais efeitos diferentes valores de K podem ter?

A métrica de disância também, apenas pegamos a distância do cosseno entre os genêros e somamos com a diferença entre a popularidade normalizada. Pode melhorar o algoritmo nisso?

In [12]:
# Utilizando diferentes valores de K para verificar qual pode chegar mais próximo

def avg_rating(K):
    '''Utilizando a função para reutilização de código apesar de estar referênciando variáveis declaradas na memória'''
    avgRating = 0
    neighbors = getNeighbors(1,K)
    for neighbor in neighbors:
        avgRating += movieDict[neighbor][3] # O elemento 3 do `dataset` de filmes representa a nota do filme.
        # print(movieDict[neighbor][0] + " " + str(movieDict[neighbor][3]))
    
    avgRating /= float(K) # Calcula a média do filme por K
    return avgRating

def show_avg_rating(k):
    print(f'{k} K: {avg_rating(k)} ; Toy Story Rating {movieDict[1][3]}')

show_avg_rating(10) # Valor usado na aula
show_avg_rating(9)
show_avg_rating(7)
show_avg_rating(5) # A quantidade de 5 vizinhos parece interessante, é o que chegou mais próximo no caso de Toy Story
show_avg_rating(3)
show_avg_rating(15)
show_avg_rating(11)
show_avg_rating(6)
show_avg_rating(4)

10 K: 3.3445905900235564 ; Toy Story Rating 3.8783185840707963
9 K: 3.3999724504535243 ; Toy Story Rating 3.8783185840707963
7 K: 3.4383535437685526 ; Toy Story Rating 3.8783185840707963
5 K: 3.7189656165466287 ; Toy Story Rating 3.8783185840707963
3 K: 3.5337960865154954 ; Toy Story Rating 3.8783185840707963
15 K: 3.466910233754485 ; Toy Story Rating 3.8783185840707963
11 K: 3.384338552913977 ; Toy Story Rating 3.8783185840707963
6 K: 3.546668877986388 ; Toy Story Rating 3.8783185840707963
4 K: 3.6669609889372543 ; Toy Story Rating 3.8783185840707963


In [27]:
# Agora alterando a função que calcula a distância

from scipy import spatial

def ComputeDistance(a, b):
    genresA = a[1]
    genresB = b[1]
    genreDistance = spatial.distance.cosine(genresA, genresB)

    genreDistance = genreDistance * 1.05 # Simulando pesos de 0 a 1, utilizando 0.05
    
    popularityA = a[2]
    popularityB = b[2]
    popularityDistance = abs(popularityA - popularityB)

    popularityDistance = popularityDistance * 01.95 # Nesse caso o peso restante é 0.95 assim podemos ter uma influência 
                                                    # maior de popularidade do que de genêro.
    
    return genreDistance + popularityDistance

ComputeDistance(movieDict[2], movieDict[4]) # Quanto mais próximo a 0 o resultado mais parecidos são os filmes.


np.float64(0.9608919382504288)

In [28]:
# Os valores de 1.05 e 1.95 trouxeram resultados interessantes utilizando 10 e 15 K.
show_avg_rating(10)
show_avg_rating(9)
show_avg_rating(7)
show_avg_rating(5)
show_avg_rating(3)
show_avg_rating(15)
show_avg_rating(11)
show_avg_rating(6)
show_avg_rating(4)

10 K: 3.716070847290196 ; Toy Story Rating 3.8783185840707963
9 K: 3.7087655878981973 ; Toy Story Rating 3.8783185840707963
7 K: 3.6961142576921135 ; Toy Story Rating 3.8783185840707963
5 K: 3.7189656165466287 ; Toy Story Rating 3.8783185840707963
3 K: 3.6183528558737215 ; Toy Story Rating 3.8783185840707963
15 K: 3.7116533967305805 ; Toy Story Rating 3.8783185840707963
11 K: 3.6908124464663996 ; Toy Story Rating 3.8783185840707963
6 K: 3.738185632836476 ; Toy Story Rating 3.8783185840707963
4 K: 3.695510673651323 ; Toy Story Rating 3.8783185840707963
